# Results analysis

Reads `eval_test.json` files produced by `src.evaluate` and the `fusion_results.json` from `src.multi_mag_fusion`.

Sections:
1. Backbone comparison table.
2. Patient vs image-level accuracy gap.
3. Per-magnification breakdown.
4. Multi-magnification fusion lift over single-mag baselines.
5. Confusion matrices for the best model.

In [ ]:
import sys, os, json, glob
sys.path.insert(0, os.path.abspath('..'))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

RESULTS = os.path.abspath('../results')
eval_files = sorted(glob.glob(os.path.join(RESULTS, '**', 'eval_test.json'), recursive=True))
print(f'found {len(eval_files)} eval bundles')
for p in eval_files: print(' -', p)

In [ ]:
# Backbone comparison table
rows = []
for p in eval_files:
    with open(p) as f: b = json.load(f)
    name = os.path.basename(os.path.dirname(p))
    rows.append({
        'run': name,
        'image_acc':       b['image_level']['accuracy'],
        'image_macro_f1':  b['image_level']['macro_f1'],
        'patient_acc':     b['patient_level']['accuracy'],
        'patient_macro_f1':b['patient_level']['macro_f1'],
        'n_patients':      b['patient_level']['n_patients'],
    })
summary = pd.DataFrame(rows).sort_values('patient_acc', ascending=False)
summary

In [ ]:
# Per-magnification breakdown for the best run
best = summary.iloc[0]['run']
with open(os.path.join(RESULTS, best, 'eval_test.json')) as f:
    bundle = json.load(f)
mag_df = pd.DataFrame(bundle['per_magnification'])
display(mag_df)

fig, ax = plt.subplots(figsize=(7, 4))
x = np.arange(len(mag_df))
ax.bar(x - 0.2, mag_df.image_acc,   0.4, label='image-level acc')
ax.bar(x + 0.2, mag_df.patient_acc, 0.4, label='patient-level acc')
ax.set_xticks(x); ax.set_xticklabels([f'{m}X' for m in mag_df.magnification])
ax.set_ylim(0, 1); ax.set_title(f'{best}: per-magnification accuracy'); ax.legend()
plt.show()

In [ ]:
# Confusion matrix at the patient level
cm = np.array(bundle['patient_level']['confusion_matrix'])
classes = bundle['patient_level']['classes']
fig, ax = plt.subplots(figsize=(0.8 * len(classes) + 2, 0.8 * len(classes) + 2))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=classes, yticklabels=classes, ax=ax)
ax.set_xlabel('predicted'); ax.set_ylabel('true')
ax.set_title(f'{best}: patient-level confusion matrix')
plt.tight_layout(); plt.show()

In [ ]:
# Multi-magnification fusion lift (if present)
fusion_path = os.path.join(RESULTS, 'fusion', 'fusion_results.json')
if os.path.exists(fusion_path):
    with open(fusion_path) as f: fr = json.load(f)
    rows = [{'name': r['name'], 'patient_acc': r['accuracy'], 'macro_f1': r['macro_f1']}
            for r in fr['per_mag_only']]
    rows += [
        {'name': 'fused_mean',     'patient_acc': fr['fused_mean']['accuracy'],     'macro_f1': fr['fused_mean']['macro_f1']},
        {'name': 'fused_weighted', 'patient_acc': fr['fused_weighted']['accuracy'], 'macro_f1': fr['fused_weighted']['macro_f1']},
        {'name': 'fused_max',      'patient_acc': fr['fused_max']['accuracy'],      'macro_f1': fr['fused_max']['macro_f1']},
    ]
    fdf = pd.DataFrame(rows)
    display(fdf)
    fdf.set_index('name')[['patient_acc', 'macro_f1']].plot.bar(figsize=(9, 4), ylim=(0, 1))
    plt.title('Single-magnification vs fused models'); plt.show()
else:
    print('Run src.multi_mag_fusion first to generate fusion_results.json')